# Nehale Smart Prepaid Meter — Usage Predictor & Top-up Recommendation
**DPJ621S · Simon E. Nehale (223019267)**

This notebook trains the model that the web app (and the ESP32) use to predict:
* how many **kWh per day** the house will use,
* how many **days the balance will last** and the run-out date,
* the **recommended top-up in N$** so the units last 30 days.

**Model** — a linear model on the same 4 features the firmware uses:

`kWh/day = c1·yesterday + c2·(2 days ago) + c3·(3 days ago) + c4·(mean of last 7 days)`

It has **no intercept**, so it is *scale-free*: a model trained on a big household (≈10–30 kWh/day) works unchanged
for the 6 W prototype load (≈0.15 kWh/day). That is why we can train on a public dataset first and on the meter's own data later.

**Steps:** choose the data source → Run all → the last cell downloads `model.json` → upload it on the web app's **Model** page.

In [ ]:
# ===== SETTINGS =====
DATA_SOURCE = "uci"        # "uci"       = public UCI household power dataset (4 years, downloads ~20 MB)
                           # "own"       = daily CSV exported from the web app (History -> Daily CSV)
                           # "synthetic" = generated data (no download, for a quick test)
TARIFF_NAD_PER_KWH = 2.0   # same as the firmware / web app
TARGET_DAYS = 30           # recommended top-up should last this many days
TEST_FRACTION = 0.2        # last 20 % of the days are kept for testing

In [ ]:
import io, json, zipfile, urllib.request, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
plt.rcParams["figure.figsize"] = (11, 3.5)

## 1. Load daily energy use (kWh per day)

In [ ]:
def load_uci():
    url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
    print("downloading", url)
    raw = urllib.request.urlopen(url).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    name = [n for n in z.namelist() if n.endswith(".txt")][0]
    df = pd.read_csv(z.open(name), sep=";", na_values="?", low_memory=False,
                     usecols=["Date", "Global_active_power"])
    df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
    # one row per minute, power in kW  ->  kWh per minute = kW / 60
    g = df.groupby("Date")["Global_active_power"]
    daily = pd.DataFrame({"kwh": g.sum() / 60.0, "minutes": g.count()})
    daily = daily[daily["minutes"] >= 0.9 * 1440]          # drop days with > 10 % missing minutes
    daily = daily.asfreq("D")                              # missing days become NaN
    return daily["kwh"], "UCI Individual household electric power consumption (France, 2006-2010)"

def load_own():
    from google.colab import files
    up = files.upload()                                    # choose NEHALE-01_daily_usage.csv
    df = pd.read_csv(io.BytesIO(list(up.values())[0]), parse_dates=["date"])
    df = df[df["valid"] == 1].set_index("date").asfreq("D")
    return df["kwh_per_day"], "Own meter data (" + list(up.keys())[0] + ")"

def load_synthetic(n=730, seed=1):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2024-01-01", periods=n, freq="D")
    t = np.arange(n)
    kwh = (0.15 * (1 + 0.25 * np.cos(2 * np.pi * (t - 180) / 365))     # winter higher
                * np.where(idx.dayofweek >= 5, 1.2, 1.0)                    # weekends higher
                * rng.normal(1, 0.08, n))
    return pd.Series(kwh, index=idx), "Synthetic 6 W household pattern"

try:
    series, SOURCE_NAME = {"uci": load_uci, "own": load_own, "synthetic": load_synthetic}[DATA_SOURCE]()
except Exception as e:
    print("Could not load", DATA_SOURCE, "data:", e, "-> using synthetic data instead")
    series, SOURCE_NAME = load_synthetic()
series.name = "kwh"
print(SOURCE_NAME)
print(f"{series.notna().sum()} usable days, mean {series.mean():.3f} kWh/day")
series.plot(title="Daily energy use"); plt.ylabel("kWh/day"); plt.show()
if DATA_SOURCE == "own" and series.notna().sum() < 30:
    print("WARNING: fewer than 30 days of own data - train on 'uci' first and use this data later.")

## 2. Features (identical to the firmware and the web app)
For every day *d*: `lag1` = use on *d-1*, `lag2` = *d-2*, `lag3` = *d-3*, `mean7` = mean of *d-1 … d-7*. Target = use on *d*.

In [ ]:
FEATURES = ["lag1", "lag2", "lag3", "mean7"]

def make_features(s):
    df = pd.DataFrame({"y": s})
    df["lag1"] = s.shift(1); df["lag2"] = s.shift(2); df["lag3"] = s.shift(3)
    df["mean7"] = s.shift(1).rolling(7).mean()
    return df.dropna()

data = make_features(series)
split = int(len(data) * (1 - TEST_FRACTION))
train, test = data.iloc[:split], data.iloc[split:]
print(f"train {len(train)} days ({train.index[0].date()} .. {train.index[-1].date()})")
print(f"test  {len(test)} days ({test.index[0].date()} .. {test.index[-1].date()})")
data.head()

## 3. Train and compare with the simple 7-day average

In [ ]:
lin = LinearRegression(fit_intercept=False).fit(train[FEATURES], train["y"])

# Random forest for comparison only (not exported). Trained on values / mean7 so it is scale-free too.
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=0)
rf.fit(train[FEATURES].div(train["mean7"], axis=0), train["y"] / train["mean7"])

pred = pd.DataFrame({
    "actual": test["y"],
    "7-day average": test["mean7"],
    "linear model": lin.predict(test[FEATURES]),
    "random forest": rf.predict(test[FEATURES].div(test["mean7"], axis=0)) * test["mean7"],
}, index=test.index)

mean_y = test["y"].mean()
results = pd.DataFrame({c: {"MAE (kWh/day)": mean_absolute_error(pred["actual"], pred[c]),
                            "MAE (% of mean)": 100 * mean_absolute_error(pred["actual"], pred[c]) / mean_y}
                        for c in pred.columns if c != "actual"}).T
print("coefficients:", {f: round(float(c), 4) for f, c in zip(FEATURES, lin.coef_)}, " sum =", round(float(lin.coef_.sum()), 4))
display(results.round(4))
pred.iloc[-90:].plot(title="Test period - last 90 days"); plt.ylabel("kWh/day"); plt.show()

## 4. Does it predict *days left* well?
For every test day we pretend the customer has a balance of about 20 days of units, forecast day by day
(each prediction feeds the next, exactly like the web app) until the balance is used up, and compare with the real number of days.

In [ ]:
def predict_next(hist, coef):
    h = np.asarray(hist)
    x = np.array([h[-1], h[-2], h[-3], h[-7:].mean()])
    return max(float(x @ coef), 0.0)

def days_left_forecast(hist, balance, coef, max_days=999):
    hist, d = list(hist), 0.0
    while d < max_days:
        p = predict_next(hist, coef)
        if p <= 1e-9: return max_days
        if balance <= p: return d + balance / p
        balance -= p; d += 1; hist.append(p)
    return max_days

def days_left_actual(future, balance):
    d = 0.0
    for u in future:
        if balance <= u: return d + balance / u
        balance -= u; d += 1
    return np.nan

s = series.ffill().values
dates = series.index
baseline_coef = np.array([0, 0, 0, 1.0])
rows = []
for i in range(split + 7, len(s) - 60):
    hist = s[i - 14:i]
    bal = 20 * hist[-7:].mean()
    rows.append({"date": dates[i],
                 "actual": days_left_actual(s[i:], bal),
                 "linear model": days_left_forecast(hist, bal, lin.coef_),
                 "7-day average": days_left_forecast(hist, bal, baseline_coef)})
dl = pd.DataFrame(rows).set_index("date").dropna()
dl_mae = {c: mean_absolute_error(dl["actual"], dl[c]) for c in ["linear model", "7-day average"]}
print("Days-left error (MAE):", {k: round(v, 2) for k, v in dl_mae.items()}, "days")
dl.iloc[:120].plot(title="Days left: forecast vs actual (balance = 20 days of average use)"); plt.ylabel("days"); plt.show()

## 5. Example recommendation (what the web page shows)

In [ ]:
balance_kwh = 1.2                                     # try your own numbers
example_hist = series.dropna().values[-14:]
scale = 0.15 / example_hist.mean()                    # rescale to the 6 W prototype load (~0.15 kWh/day)
hist = list(example_hist * scale)

days = days_left_forecast(hist, balance_kwh, lin.coef_)
h, rates = list(hist), []
for _ in range(TARGET_DAYS):
    p = predict_next(h, lin.coef_); rates.append(p); h.append(p)
need = max(sum(rates) - balance_kwh, 0)
today = dt.date.today()
print(f"Forecast use      : {rates[0]:.3f} kWh/day")
print(f"Balance           : {balance_kwh:.2f} kWh")
print(f"Days left         : {days:.1f}  -> runs out {today + dt.timedelta(days=days):%a %d %b %Y}")
print(f"Top up by         : {today + dt.timedelta(days=max(days - 2, 0)):%a %d %b %Y}")
print(f"Recommended top-up: N$ {np.ceil(need * TARIFF_NAD_PER_KWH):.0f}  ({need:.2f} kWh for {TARGET_DAYS} days)")

## 6. Export `model.json` → upload on the web app's **Model** page

In [ ]:
lin_mae = results.loc["linear model", "MAE (% of mean)"] / 100
base_mae = results.loc["7-day average", "MAE (% of mean)"] / 100
model_info = {
    "name": "Linear lag model",
    "type": "linear",
    "features": FEATURES,
    "intercept": 0.0,
    "coef": [round(float(c), 6) for c in lin.coef_],
    "trained_on": SOURCE_NAME,
    "trained_at": dt.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "train_days": int(len(train)),
    "test_mae_rel": round(float(lin_mae), 4),
    "baseline_mae_rel": round(float(base_mae), 4),
    "days_left_mae": round(float(dl_mae["linear model"]), 3),
}
with open("model.json", "w") as f:
    json.dump(model_info, f, indent=2)
print(json.dumps(model_info, indent=2))

c = lin.coef_
print("\nFor the ESP32 firmware (section 1, 'Usage model'):")
print(f"const float MB0 = 0.0, MB1 = {c[0]:.4f}, MB2 = {c[1]:.4f}, MB3 = {c[2]:.4f}, MB4 = {c[3]:.4f};")

try:
    from google.colab import files
    files.download("model.json")
except ImportError:
    print("(not in Colab - model.json saved next to the notebook)")